# 01 - Train a custom baseline (U-Net / MedNeXt / DA-VNet / AKS-UNet)

Runtime -> Change runtime type -> **T4 GPU**. Each team member sets `MODEL` to their model.
Training checkpoints to Drive every epoch; if Colab disconnects or units run out, **just re-run all cells** and it
resumes. Set `MAX_HOURS` a little below your remaining session so it stops cleanly.

In [ ]:
MODEL = 'mednext'      # 'unet3d' | 'mednext' | 'davnet' | 'aksunet'
SMOKE = True           # True: 3-epoch timing run into runs/smoke_<model>. False: the real run
EPOCHS = 200           # x 250 iterations; set from the smoke-test timing, identical across models
MAX_HOURS = 3.5
PATCH = '192 192 16'   # H W D(slices). On OOM: '160 160 16' or BATCH = 1
BATCH = 2

In [ ]:
# --- shared setup: Drive + code ---------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/figo'          # shared project folder (one Colab account)
REPO_URL = 'https://github.com/acm-research-f26/Radii.git'  # public; if made private see README (token in Colab Secrets)
REPO = '/content/figo-repo'

import os, subprocess
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
%cd {REPO}
!pip -q install -r requirements.txt

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# copy the cache to local disk once per session (fast random access during training)
!rsync -a {DRIVE}/data/cache_0.9mm/ /content/cache/
RUN = f'{DRIVE}/runs/' + (f'smoke_{MODEL}' if SMOKE else MODEL)
if SMOKE:
    EPOCHS = 3

In [ ]:
!python -m figo.train --model {MODEL} --cache-dir /content/cache --splits {DRIVE}/data/splits.json \
    --out-dir {RUN} --epochs {EPOCHS} --max-hours {MAX_HOURS} --patch-size {PATCH} --batch-size {BATCH}

In [ ]:
# timing -> how many epochs fit the budget (row 1 = pure training, row 2 = training + validation)
import csv
log = list(csv.DictReader(open(f'{RUN}/log.csv')))
for r in log:
    print('epoch', r['epoch'], r['epoch_sec'], 's')
if len(log) >= 3:
    t_train = float(log[1]['epoch_sec'])
    t_val = max(float(log[2]['epoch_sec']) - t_train, 0)
    per_epoch = t_train + t_val / 10          # validation runs every 10 epochs
    print(f'~{per_epoch:.0f} s per epoch on average; validation ~{t_val:.0f} s')
    for h in (6, 8, 10):
        print(f'  {h:2d} GPU-hours  ->  EPOCHS = {int(h * 3600 / per_epoch)}')
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
# learning curves
import csv, matplotlib.pyplot as plt
log = list(csv.DictReader(open(f'{RUN}/log.csv')))
plt.plot([float(r['train_loss']) for r in log], label='train loss')
v = [(int(r['epoch']), float(r['val_mean'])) for r in log if r['val_mean']]
if v: plt.plot(*zip(*v), 'o-', label='val mean Dice')
plt.legend(); plt.xlabel('epoch'); plt.show()

## Test-set inference and evaluation (run after training has finished)

In [ ]:
PRED = f'{DRIVE}/preds/{MODEL}'
RES = f'{DRIVE}/results'
!python -m figo.infer --checkpoint {RUN}/best.pt --data-dir {DRIVE}/data/clean --splits {DRIVE}/data/splits.json \
    --subset test --out-dir {PRED}
!python -m figo.evaluate --pred-dir {PRED} --data-dir {DRIVE}/data/clean --splits {DRIVE}/data/splits.json \
    --subset test --out {RES}/{MODEL}_seg.csv
!python -m figo.figo_eval agreement --ref-dir {DRIVE}/data/clean --pred-dir {PRED} \
    --splits {DRIVE}/data/splits.json --subset test --out {RES}/{MODEL}_figo_agreement.csv